# Oracle Table Copy Flow

### Copia manual Oracle -> Oracle (1 a N tablas)

Este notebook ejecuta una copia **completa** por tabla, aplicando respaldo opcional, recreacion de estructura y validacion final por conteo de filas.

## Ruta de ejecucion
1. Configurar constantes de conexion y parametros.
2. Importar dependencias y preparar utilidades.
3. Ejecutar funciones de backup, drop/create y carga por lotes.
4. Revisar validaciones y resultado consolidado.

## 1) Configuracion manual (constantes en MAYUS_SNAKE_CASE)

Edita esta celda antes de ejecutar. Aqui defines origen, destino y parametros del flujo en constantes.

> Nota: si tu conexion usa SID, completa `*_SID`. Si usa SERVICE_NAME, completa `*_SERVICE_NAME`.

In [1]:
# ====== ORIGEN ======
# ! Dev
# ORACLE_DST_HOST = '172.19.0.42'
# ORACLE_DST_PORT = '1521'
# ORACLE_DST_SID = 'INABIF02'
# ORACLE_DST_SERVICE_NAME = None
# ORACLE_DST_USER = 'devuser'
# ORACLE_DST_PWD = 'D3vUs3r$2025'

# ! Test
# ORACLE_SRC_HOST = '172.19.0.42'
# ORACLE_SRC_PORT = '1521'
# ORACLE_SRC_SID = 'INABIF02'
# ORACLE_SRC_SERVICE_NAME = None
# ORACLE_SRC_USER = 'USRSEGURIDAD'
# ORACLE_SRC_PWD = 'C0ntr4s3n4$3gur4'

# ! Prod
ORACLE_SRC_HOST = '172.19.0.9'
ORACLE_SRC_PORT = '1521'
ORACLE_SRC_SID = 'INABIF02'
ORACLE_SRC_SERVICE_NAME = None
ORACLE_SRC_USER = 'USRSEGURIDAD'
ORACLE_SRC_PWD = 'QL7nstYOMwxQ'


# ====== DESTINO ======
# ! Dev
ORACLE_DST_HOST = '172.19.0.9'
ORACLE_DST_PORT = '1521'
ORACLE_DST_SID = 'INABIF02'
ORACLE_DST_SERVICE_NAME = None
ORACLE_DST_USER = 'USRSEGURIDAD'
ORACLE_DST_PWD = 'QL7nstYOMwxQ'

# Catalogo de acciones (el usuario solo envia la key)
ACTION_CATALOG = {
   '1': 'COPY_DATA',
   '2': 'DDL_TABLES_FULL',
   '3': 'DDL_STORED_CODE',
   '4': 'DDL_CONSTRAINTS',
   '5': 'DDL_SEQUENCES',
   '6': 'EXPORT_TABLES_MD'
}

# Acciones a ejecutar en orden. Default: mantiene flujo actual.
ACTION_KEYS = ['6']

# Tablas objetivo para acciones: COPY_DATA, DDL_TABLES_FULL, DDL_CONSTRAINTS y DDL_SEQUENCES
TABLES_TO_COPY = [
   'SSI_ANEXOS_CABECERA',
   # 'TGPERSONA',
   # 'TGUNIDADORGANICA',
   # 'TGUBIGEO',
]

# Objetos PL/SQL a copiar por nombre (sin ambiguedad en el esquema).
STORED_OBJECT_NAMES = [
   # 'PRC_EJEMPLO',
   # 'FNC_EJEMPLO'
]

# Export markdown de todas las tablas del esquema origen.
EXPORT_MD_PATH = '../plsql_scripts/oracle_schema_tables_catalog.md'
EXPORT_INCLUDE_SUMMARY = True

# Constantes de ejecucion para COPY_DATA
BATCH_SIZE = 10000
RUN_BACKUP_BEFORE_DROP = True

## 2) Dependencias y utilidades

Si falta `oracledb`, instalalo con `pip install oracledb` y vuelve a ejecutar esta seccion.

In [2]:
from __future__ import annotations

import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple

import oracledb

PANDAS_AVAILABLE = True
try:
    import pandas as pd
except Exception as ex:
    PANDAS_AVAILABLE = False
    pd = None
    print('Aviso: pandas no se pudo importar. Se usara reporte en lista/dict.')
    print(f'Detalle: {ex}')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger('oracle_copy_flow')

def show_runtime_diagnostics() -> None:
    import os
    import sys
    print('--- Runtime Diagnostics ---')
    print(f'cwd: {os.getcwd()}')
    print(f'python: {sys.executable}')
    print(f'version: {sys.version.splitlines()[0]}')
    print(f'pandas_available: {PANDAS_AVAILABLE}')

def make_dsn(host: str, port: str, sid: str | None = None, service_name: str | None = None) -> str:
    port_num = int(port)
    if sid:
        return oracledb.makedsn(host=host, port=port_num, sid=sid)
    if service_name:
        return oracledb.makedsn(host=host, port=port_num, service_name=service_name)
    raise ValueError('Debes definir SID o SERVICE_NAME para la conexion Oracle.')

def get_connection(host: str, port: str, user: str, pwd: str, sid: str | None = None, service_name: str | None = None):
    dsn = make_dsn(host=host, port=port, sid=sid, service_name=service_name)
    return oracledb.connect(user=user, password=pwd, dsn=dsn)

def table_exists(conn, table_name: str, owner: str) -> bool:
    sql = (
        'SELECT COUNT(1) FROM ALL_TABLES '
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name'
    )
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
        return cur.fetchone()[0] > 0

def quote_ident(name: str) -> str:
    return f'"{name.upper()}"'

def get_columns_metadata(conn, owner: str, table_name: str) -> List[Tuple]:
    sql = (
        'SELECT COLUMN_NAME, DATA_TYPE, DATA_LENGTH, DATA_PRECISION, DATA_SCALE, NULLABLE '
        'FROM ALL_TAB_COLUMNS '
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name '
        'ORDER BY COLUMN_ID'
    )
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
        rows = cur.fetchall()
    if not rows:
        raise ValueError(f'No se encontro metadata para {owner}.{table_name}')
    return rows

EXCLUDED_BINARY_TYPES = {'BLOB', 'RAW', 'LONG RAW'}

def is_excluded_column_type(data_type: str) -> bool:
    t = data_type.upper()
    return t == 'ROWID' or t in EXCLUDED_BINARY_TYPES

def get_excluded_columns(cols: List[Tuple]) -> List[str]:
    return [c[0] for c in cols if is_excluded_column_type(c[1])]

def normalize_action_keys(action_keys: List[str]) -> List[str]:
    normalized = []
    for key in action_keys:
        key = str(key).strip()
        if not key:
            continue
        if key not in ACTION_CATALOG:
            raise ValueError(f'Accion no soportada: {key}. Catalogo valido: {sorted(ACTION_CATALOG.keys())}')
        normalized.append(key)
    if not normalized:
        raise ValueError('Debes definir al menos una accion en ACTION_KEYS.')
    return normalized

def get_action_name(action_key: str) -> str:
    return ACTION_CATALOG[action_key]

def get_table_list_or_fail() -> List[str]:
    tables = [t.strip().upper() for t in TABLES_TO_COPY if t and t.strip()]
    if not tables:
        raise ValueError('Debes definir TABLES_TO_COPY con al menos una tabla para esta accion.')
    return tables

def get_stored_objects_or_fail() -> List[str]:
    names = [n.strip().upper() for n in STORED_OBJECT_NAMES if n and n.strip()]
    if not names:
        raise ValueError('Debes definir STORED_OBJECT_NAMES con al menos un nombre para DDL_STORED_CODE.')
    return names

def run_ddl_query(conn, ddl_sql: str, params: Dict[str, str]) -> str:
    with conn.cursor() as cur:
        cur.execute(ddl_sql, params)
        row = cur.fetchone()
    if not row or not row[0]:
        raise ValueError(f'No se obtuvo DDL para parametros: {params}')
    return row[0].read() if hasattr(row[0], 'read') else str(row[0])

def set_metadata_transforms(conn) -> None:
    sql = (
        'BEGIN '
        "DBMS_METADATA.SET_TRANSFORM_PARAM(DBMS_METADATA.SESSION_TRANSFORM, 'STORAGE', FALSE); "
        "DBMS_METADATA.SET_TRANSFORM_PARAM(DBMS_METADATA.SESSION_TRANSFORM, 'SEGMENT_ATTRIBUTES', FALSE); "
        "DBMS_METADATA.SET_TRANSFORM_PARAM(DBMS_METADATA.SESSION_TRANSFORM, 'SQLTERMINATOR', TRUE); "
        "DBMS_METADATA.SET_TRANSFORM_PARAM(DBMS_METADATA.SESSION_TRANSFORM, 'PRETTY', TRUE); "
        'END;'
    )
    with conn.cursor() as cur:
        cur.execute(sql)

def get_table_ddl(src_conn, owner: str, table_name: str) -> str:
    sql = 'SELECT DBMS_METADATA.GET_DDL(\'TABLE\', :name, :owner) FROM DUAL'
    return run_ddl_query(src_conn, sql, {'name': table_name.upper(), 'owner': owner.upper()})

def get_sequence_ddl(src_conn, owner: str, sequence_name: str) -> str:
    sql = 'SELECT DBMS_METADATA.GET_DDL(\'SEQUENCE\', :name, :owner) FROM DUAL'
    return run_ddl_query(src_conn, sql, {'name': sequence_name.upper(), 'owner': owner.upper()})

def get_object_ddl(src_conn, owner: str, object_type: str, object_name: str) -> str:
    sql = 'SELECT DBMS_METADATA.GET_DDL(:obj_type, :name, :owner) FROM DUAL'
    return run_ddl_query(src_conn, sql, {'obj_type': object_type.upper(), 'name': object_name.upper(), 'owner': owner.upper()})

def apply_ddl(dst_conn, ddl_text: str) -> None:
    with dst_conn.cursor() as cur:
        cur.execute(ddl_text)
    dst_conn.commit()

def get_primary_unique_check_constraints(src_conn, owner: str, table_name: str) -> List[str]:
    sql = (
        'SELECT CONSTRAINT_NAME FROM ALL_CONSTRAINTS '
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name AND CONSTRAINT_TYPE IN (\'P\', \'U\', \'C\') '
        'ORDER BY DECODE(CONSTRAINT_TYPE, \'P\', 1, \'U\', 2, \'C\', 3), CONSTRAINT_NAME'
    )
    with src_conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
        return [r[0] for r in cur.fetchall()]

def get_fk_constraints(src_conn, owner: str, table_name: str) -> List[str]:
    sql = (
        'SELECT CONSTRAINT_NAME FROM ALL_CONSTRAINTS '
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name AND CONSTRAINT_TYPE = \'R\' '
        'ORDER BY CONSTRAINT_NAME'
    )
    with src_conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
        return [r[0] for r in cur.fetchall()]

def get_constraint_ddl(src_conn, owner: str, constraint_name: str) -> str:
    sql = 'SELECT DBMS_METADATA.GET_DDL(\'CONSTRAINT\', :name, :owner) FROM DUAL'
    return run_ddl_query(src_conn, sql, {'name': constraint_name.upper(), 'owner': owner.upper()})

def get_sequences_for_tables(src_conn, owner: str, table_names: List[str]) -> List[str]:
    sql = (
        'SELECT DISTINCT REGEXP_SUBSTR(DATA_DEFAULT, ''([A-Za-z0-9_$#]+)\\.NEXTVAL'', 1, 1, \'i\', 1) AS SEQ_NAME '
        'FROM ALL_TAB_COLUMNS '
        'WHERE OWNER = :owner AND TABLE_NAME = :table_name AND DATA_DEFAULT IS NOT NULL'
    )
    seq_names = set()
    with src_conn.cursor() as cur:
        for table_name in table_names:
            cur.execute(sql, owner=owner.upper(), table_name=table_name.upper())
            for row in cur.fetchall():
                if row[0]:
                    seq_names.add(row[0].upper())
    return sorted(seq_names)

def materialize_row(row: tuple) -> tuple:
    out = []
    for val in row:
        if val is None:
            out.append(None)
        elif hasattr(val, 'read'):
            out.append(val.read())
        else:
            out.append(val)
    return tuple(out)

def map_oracle_type_to_bind(data_type: str):
    t = data_type.upper()
    if t in ('VARCHAR2', 'NVARCHAR2', 'CHAR', 'NCHAR'):
        return oracledb.DB_TYPE_VARCHAR
    if t == 'NUMBER':
        return oracledb.DB_TYPE_NUMBER
    if t == 'DATE':
        return oracledb.DB_TYPE_DATE
    if t.startswith('TIMESTAMP'):
        return oracledb.DB_TYPE_TIMESTAMP
    if t in ('CLOB', 'NCLOB', 'LONG'):
        return oracledb.DB_TYPE_CLOB
    if t == 'BLOB':
        return oracledb.DB_TYPE_BLOB
    if t == 'RAW':
        return oracledb.DB_TYPE_RAW
    return oracledb.DB_TYPE_VARCHAR

def copy_table_data(src_conn, dst_conn, src_owner: str, dst_owner: str, table_name: str, batch_size: int = 10000) -> int:
    cols = get_columns_metadata(src_conn, src_owner, table_name)
    copyable_cols = [c for c in cols if not is_excluded_column_type(c[1])]
    copyable_names = [c[0] for c in copyable_cols]
    excluded_names = [c[0] for c in cols if is_excluded_column_type(c[1])]
    logger.info('Tabla %s: columnas origen=%s, copiables=%s, excluidas=%s', table_name, len(cols), len(copyable_names), len(excluded_names))
    if excluded_names:
        logger.info('Tabla %s: columnas excluidas=%s', table_name, ', '.join(excluded_names))
    if not copyable_names:
        raise ValueError(f'Tabla {table_name} no tiene columnas copiables.')

    bind_types = [map_oracle_type_to_bind(c[1]) for c in copyable_cols]

    src_owner_q = quote_ident(src_owner)
    dst_owner_q = quote_ident(dst_owner)
    table_q = quote_ident(table_name)
    col_list = ', '.join(quote_ident(c) for c in copyable_names)
    binds = ', '.join([f':{i}' for i in range(1, len(copyable_names) + 1)])

    select_sql = f'SELECT {col_list} FROM {src_owner_q}.{table_q}'
    insert_sql = f'INSERT INTO {dst_owner_q}.{table_q} ({col_list}) VALUES ({binds})'

    total_inserted = 0
    with src_conn.cursor() as src_cur, dst_conn.cursor() as dst_cur:
        src_cur.execute(select_sql)
        dst_cur.setinputsizes(*bind_types)
        while True:
            rows = src_cur.fetchmany(batch_size)
            if not rows:
                break
            materialized = [materialize_row(r) for r in rows]
            dst_cur.executemany(insert_sql, materialized)
            total_inserted += len(materialized)

    dst_conn.commit()
    logger.info('Filas insertadas en %s: %s', table_name, total_inserted)
    return total_inserted

def run_table_copy(src_conn, dst_conn, table_name: str, run_backup: bool, batch_size: int) -> Dict:
    src_owner = ORACLE_SRC_USER.upper()
    dst_owner = ORACLE_DST_USER.upper()

    result = {
        'table_name': table_name.upper(),
        'src_host': f"{ORACLE_SRC_HOST}:{ORACLE_SRC_PORT}",
        'src_user': ORACLE_SRC_USER,
        'dst_host': f"{ORACLE_DST_HOST}:{ORACLE_DST_PORT}",
        'dst_user': ORACLE_DST_USER,
        'status': 'OK',
        'backup_table': '',
        'src_count': 0,
        'dst_count': 0,
        'inserted_rows': 0,
        'excluded_columns': '',
        'message': ''
    }

    try:
        if run_backup:
            result['backup_table'] = backup_table_if_exists(dst_conn, dst_owner, table_name)

        cols = get_columns_metadata(src_conn, src_owner, table_name)
        excluded_cols = get_excluded_columns(cols)
        result['excluded_columns'] = ', '.join(excluded_cols) if excluded_cols else ''

        drop_table_if_exists(dst_conn, dst_owner, table_name)
        create_empty_table_like_source(src_conn, dst_conn, src_owner, dst_owner, table_name)

        inserted = copy_table_data(src_conn, dst_conn, src_owner, dst_owner, table_name, batch_size=batch_size)
        src_count = count_rows(src_conn, src_owner, table_name)
        dst_count = count_rows(dst_conn, dst_owner, table_name)

        result['inserted_rows'] = inserted
        result['src_count'] = src_count
        result['dst_count'] = dst_count

        if src_count != dst_count:
            raise RuntimeError(f'Validacion fallida en {table_name}: origen={src_count}, destino={dst_count}')

        result['message'] = 'Copia completa y validada.'
    except Exception as ex:
        dst_conn.rollback()
        result['status'] = 'ERROR'
        result['message'] = str(ex)
        logger.exception('Error copiando tabla %s', table_name)

    return result

def count_rows(conn, owner: str, table_name: str) -> int:
    owner_q = quote_ident(owner)
    table_q = quote_ident(table_name)
    sql = f'SELECT COUNT(1) FROM {owner_q}.{table_q}'
    with conn.cursor() as cur:
        cur.execute(sql)
        return int(cur.fetchone()[0])

def export_tables_catalog_md(src_conn, owner: str, md_path: str, include_summary: bool = True) -> Dict:
    sql_tables = 'SELECT TABLE_NAME FROM ALL_TABLES WHERE OWNER = :owner ORDER BY TABLE_NAME'
    sql_cols = (
        'SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, DATA_LENGTH, DATA_PRECISION, DATA_SCALE, NULLABLE, DATA_DEFAULT '
        'FROM ALL_TAB_COLUMNS WHERE OWNER = :owner ORDER BY TABLE_NAME, COLUMN_ID'
    )
    with src_conn.cursor() as cur:
        cur.execute(sql_tables, owner=owner.upper())
        tables = [r[0] for r in cur.fetchall()]
        cur.execute(sql_cols, owner=owner.upper())
        cols = cur.fetchall()

    by_table: Dict[str, List[Tuple]] = {}
    type_count: Dict[str, int] = {}
    for row in cols:
        table_name = row[0]
        by_table.setdefault(table_name, []).append(row)
        data_type = row[2] or 'UNKNOWN'
        type_count[data_type] = type_count.get(data_type, 0) + 1

    lines = []
    lines.append(f'# Catalogo de tablas Oracle - {owner.upper()}')
    lines.append('')
    lines.append(f'- Fecha generacion: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    lines.append(f'- Esquema origen: {owner.upper()}')
    lines.append('')

    if include_summary:
        lines.append('## Resumen')
        lines.append('')
        lines.append(f'- Total tablas: {len(tables)}')
        lines.append(f'- Total columnas: {len(cols)}')
        lines.append('')
        lines.append('### Distribucion por tipo de dato')
        lines.append('')
        lines.append('| Tipo | Cantidad |')
        lines.append('|---|---:|')
        for data_type in sorted(type_count.keys()):
            lines.append(f'| {data_type} | {type_count[data_type]} |')
        lines.append('')

    lines.append('## Detalle por tabla')
    lines.append('')
    for table_name in tables:
        lines.append(f'### {table_name}')
        lines.append('')
        lines.append('| Columna | Tipo | Longitud | Precision | Scale | Nullable | Default |')
        lines.append('|---|---|---:|---:|---:|---|---|')
        for _, col_name, data_type, data_length, data_precision, data_scale, nullable, data_default in by_table.get(table_name, []):
            default_txt = '' if data_default is None else str(data_default).replace('\n', ' ').strip()
            lines.append(
                f"| {col_name} | {data_type} | {data_length or ''} | {data_precision or ''} | {data_scale or ''} | {nullable} | {default_txt} |"
            )
        lines.append('')

    output = '\n'.join(lines).rstrip() + '\n'
    out_path = Path(md_path).expanduser().resolve()
    out_path.write_text(output, encoding='utf-8')
    logger.info('Markdown exportado: %s', out_path)
    return {
        'action': 'EXPORT_TABLES_MD',
        'status': 'OK',
        'tables': len(tables),
        'columns': len(cols),
        'output_file': str(out_path),
        'message': 'Export completo generado.'
    }


## 2.1) Diagnostico rapido del kernel

Ejecuta esta celda si tienes errores de import para confirmar el entorno activo de Jupyter.

In [10]:
show_runtime_diagnostics()

--- Runtime Diagnostics ---
cwd: c:\work-space\sigeif-app\sigeif_ssi_inabif_plsql\py_notebooks
python: c:\Users\uti.rguevara_01\AppData\Local\Python\pythoncore-3.14-64\python.exe
version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
pandas_available: True


## 3) Motor de copia: backup, reemplazo y carga

Esta seccion aplica el flujo completo por tabla: **backup opcional**, **drop si existe**, **create nuevo** y **carga total por lotes**.

In [3]:
def backup_table_if_exists(dst_conn, owner: str, table_name: str) -> str:
    if not table_exists(dst_conn, table_name, owner):
        logger.info('No existe %s.%s en destino; no se crea backup.', owner, table_name)
        return ''

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_name = f'{table_name.upper()}_BKP_{ts}'
    owner_q = quote_ident(owner)
    src_q = quote_ident(table_name)
    bkp_q = quote_ident(backup_name)

    sql = f'CREATE TABLE {owner_q}.{bkp_q} AS SELECT * FROM {owner_q}.{src_q}'
    with dst_conn.cursor() as cur:
        cur.execute(sql)
    dst_conn.commit()
    logger.info('Backup creado: %s.%s', owner, backup_name)
    return backup_name

def drop_table_if_exists(dst_conn, owner: str, table_name: str) -> None:
    if not table_exists(dst_conn, table_name, owner):
        return

    owner_q = quote_ident(owner)
    table_q = quote_ident(table_name)
    sql = f'DROP TABLE {owner_q}.{table_q} PURGE'
    with dst_conn.cursor() as cur:
        cur.execute(sql)
    dst_conn.commit()
    logger.info('Tabla eliminada en destino: %s.%s', owner, table_name)

def create_empty_table_like_source(src_conn, dst_conn, src_owner: str, dst_owner: str, table_name: str) -> None:
    ddl = get_table_ddl(src_conn, src_owner, table_name)
    ddl = ddl.replace(f'\"{src_owner.upper()}\".', f'\"{dst_owner.upper()}\".')
    with dst_conn.cursor() as cur:
        cur.execute(ddl)
    dst_conn.commit()
    logger.info('Tabla creada en destino: %s.%s', dst_owner, table_name)

def create_table_structure_full(src_conn, dst_conn, src_owner: str, dst_owner: str, table_name: str) -> Dict:
    result = {'action': 'DDL_TABLES_FULL', 'object_type': 'TABLE', 'object_name': table_name.upper(), 'status': 'OK', 'message': ''}
    try:
        drop_table_if_exists(dst_conn, dst_owner, table_name)
        create_empty_table_like_source(src_conn, dst_conn, src_owner, dst_owner, table_name)
        result['message'] = 'Estructura creada con DDL completo.'
    except Exception as ex:
        dst_conn.rollback()
        result['status'] = 'ERROR'
        result['message'] = str(ex)
        logger.exception('Error creando estructura de tabla %s', table_name)
    return result

def create_constraints_for_table(src_conn, dst_conn, src_owner: str, dst_owner: str, table_name: str) -> List[Dict]:
    out = []
    names = get_primary_unique_check_constraints(src_conn, src_owner, table_name) + get_fk_constraints(src_conn, src_owner, table_name)
    for constraint_name in names:
        item = {'action': 'DDL_CONSTRAINTS', 'object_type': 'CONSTRAINT', 'object_name': constraint_name, 'table_name': table_name.upper(), 'status': 'OK', 'message': ''}
        try:
            ddl = get_constraint_ddl(src_conn, src_owner, constraint_name)
            ddl = ddl.replace(f'\"{src_owner.upper()}\".', f'\"{dst_owner.upper()}\".')
            apply_ddl(dst_conn, ddl)
            item['message'] = 'Constraint creado.'
        except Exception as ex:
            dst_conn.rollback()
            item['status'] = 'ERROR'
            item['message'] = str(ex)
            logger.exception('Error creando constraint %s', constraint_name)
        out.append(item)
    return out

def create_sequences_for_tables(src_conn, dst_conn, src_owner: str, dst_owner: str, table_names: List[str]) -> List[Dict]:
    out = []
    seq_names = get_sequences_for_tables(src_conn, src_owner, table_names)
    if not seq_names:
        return [{'action': 'DDL_SEQUENCES', 'object_type': 'SEQUENCE', 'object_name': '', 'status': 'OK', 'message': 'No se detectaron secuencias por NEXTVAL en defaults.'}]
    for seq_name in seq_names:
        item = {'action': 'DDL_SEQUENCES', 'object_type': 'SEQUENCE', 'object_name': seq_name, 'status': 'OK', 'message': ''}
        try:
            ddl = get_sequence_ddl(src_conn, src_owner, seq_name)
            ddl = ddl.replace(f'\"{src_owner.upper()}\".', f'\"{dst_owner.upper()}\".')
            apply_ddl(dst_conn, ddl)
            item['message'] = 'Sequence creada.'
        except Exception as ex:
            dst_conn.rollback()
            item['status'] = 'ERROR'
            item['message'] = str(ex)
            logger.exception('Error creando sequence %s', seq_name)
        out.append(item)
    return out

def resolve_object_type(src_conn, owner: str, object_name: str) -> str:
    sql = (
        'SELECT OBJECT_TYPE FROM ALL_OBJECTS '
        'WHERE OWNER = :owner AND OBJECT_NAME = :object_name '
        'AND OBJECT_TYPE IN (\'PROCEDURE\', \'FUNCTION\', \'PACKAGE\', \'VIEW\', \'TRIGGER\')'
    )
    with src_conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper(), object_name=object_name.upper())
        rows = [r[0] for r in cur.fetchall()]
    if not rows:
        raise ValueError(f'No se encontro objeto en origen: {object_name}')
    if len(rows) > 1:
        raise ValueError(f'Objeto ambiguo: {object_name}. Tipos encontrados: {rows}')
    return rows[0]

def create_stored_object(src_conn, dst_conn, src_owner: str, dst_owner: str, object_name: str) -> Dict:
    result = {'action': 'DDL_STORED_CODE', 'object_type': '', 'object_name': object_name.upper(), 'status': 'OK', 'message': ''}
    try:
        obj_type = resolve_object_type(src_conn, src_owner, object_name)
        result['object_type'] = obj_type
        ddl = get_object_ddl(src_conn, src_owner, obj_type, object_name)
        ddl = ddl.replace(f'\"{src_owner.upper()}\".', f'\"{dst_owner.upper()}\".')
        apply_ddl(dst_conn, ddl)
        result['message'] = 'Objeto PL/SQL creado.'
    except Exception as ex:
        dst_conn.rollback()
        result['status'] = 'ERROR'
        result['message'] = str(ex)
        logger.exception('Error creando objeto almacenado %s', object_name)
    return result

## 4) Ejecucion manual (paso principal)

Se ejecutan acciones segun `ACTION_KEYS` en orden. Si un objeto falla, el proceso continua y registra el error en el reporte final.

In [4]:
results = []
src_conn = None
dst_conn = None

try:
    selected_keys = normalize_action_keys(ACTION_KEYS)
    selected_actions = [get_action_name(k) for k in selected_keys]

    src_conn = get_connection(
        host=ORACLE_SRC_HOST,
        port=ORACLE_SRC_PORT,
        user=ORACLE_SRC_USER,
        pwd=ORACLE_SRC_PWD,
        sid=ORACLE_SRC_SID,
        service_name=ORACLE_SRC_SERVICE_NAME
    )
    dst_conn = get_connection(
        host=ORACLE_DST_HOST,
        port=ORACLE_DST_PORT,
        user=ORACLE_DST_USER,
        pwd=ORACLE_DST_PWD,
        sid=ORACLE_DST_SID,
        service_name=ORACLE_DST_SERVICE_NAME
    )

    metadata_actions = {'COPY_DATA', 'DDL_TABLES_FULL', 'DDL_CONSTRAINTS', 'DDL_SEQUENCES', 'DDL_STORED_CODE'}
    if any(action in metadata_actions for action in selected_actions):
        set_metadata_transforms(src_conn)

    src_owner = ORACLE_SRC_USER.upper()
    dst_owner = ORACLE_DST_USER.upper()

    logger.info('Acciones seleccionadas: %s', selected_actions)

    for action_name in selected_actions:
        logger.info('Iniciando accion: %s', action_name)

        if action_name == 'COPY_DATA':
            tables = get_table_list_or_fail()
            for table_name in tables:
                logger.info('Procesando tabla: %s', table_name)
                res = run_table_copy(
                    src_conn=src_conn,
                    dst_conn=dst_conn,
                    table_name=table_name,
                    run_backup=RUN_BACKUP_BEFORE_DROP,
                    batch_size=BATCH_SIZE
                )
                res['action'] = action_name
                results.append(res)

        elif action_name == 'DDL_TABLES_FULL':
            tables = get_table_list_or_fail()
            for table_name in tables:
                logger.info('Creando estructura full: %s', table_name)
                results.append(create_table_structure_full(src_conn, dst_conn, src_owner, dst_owner, table_name))

        elif action_name == 'DDL_CONSTRAINTS':
            tables = get_table_list_or_fail()
            for table_name in tables:
                logger.info('Creando constraints: %s', table_name)
                results.extend(create_constraints_for_table(src_conn, dst_conn, src_owner, dst_owner, table_name))

        elif action_name == 'DDL_SEQUENCES':
            tables = get_table_list_or_fail()
            logger.info('Creando secuencias asociadas a tablas objetivo')
            results.extend(create_sequences_for_tables(src_conn, dst_conn, src_owner, dst_owner, tables))

        elif action_name == 'DDL_STORED_CODE':
            object_names = get_stored_objects_or_fail()
            for object_name in object_names:
                logger.info('Creando objeto almacenado: %s', object_name)
                results.append(create_stored_object(src_conn, dst_conn, src_owner, dst_owner, object_name))

        elif action_name == 'EXPORT_TABLES_MD':
            logger.info('Exportando catalogo markdown de todas las tablas del esquema origen')
            results.append(
                export_tables_catalog_md(
                    src_conn=src_conn,
                    owner=src_owner,
                    md_path=EXPORT_MD_PATH,
                    include_summary=EXPORT_INCLUDE_SUMMARY
                )
            )

finally:
    if src_conn is not None:
        src_conn.close()
    if dst_conn is not None:
        dst_conn.close()

2026-07-19 19:54:12,997 | INFO | Acciones seleccionadas: ['EXPORT_TABLES_MD']
2026-07-19 19:54:12,999 | INFO | Iniciando accion: EXPORT_TABLES_MD
2026-07-19 19:54:13,000 | INFO | Exportando catalogo markdown de todas las tablas del esquema origen
2026-07-19 19:54:13,595 | INFO | Markdown exportado: C:\work-space\sigeif-app\sigeif_ssi_inabif_plsql\plsql_scripts\oracle_schema_tables_catalog.md


## 5) Reporte final y lectura rapida

Revisa estado por tabla, conteos origen/destino, filas insertadas y nombre de backup cuando aplique.

In [5]:
# Encabezado de conexion
print('=' * 60)
print('ORIGEN'.center(60))
print('=' * 60)
print(f"  Host : {ORACLE_SRC_HOST}:{ORACLE_SRC_PORT}")
print(f"  User : {ORACLE_SRC_USER}")
print(f"  SID  : {ORACLE_SRC_SID or ORACLE_SRC_SERVICE_NAME}")
print()
print('=' * 60)
print('DESTINO'.center(60))
print('=' * 60)
print(f"  Host : {ORACLE_DST_HOST}:{ORACLE_DST_PORT}")
print(f"  User : {ORACLE_DST_USER}")
print(f"  SID  : {ORACLE_DST_SID or ORACLE_DST_SERVICE_NAME}")
print()
print(f'Acciones solicitadas: {ACTION_KEYS}')
print(f'Total resultados: {len(results)}')
print()

if not results:
    print('No hay resultados para mostrar.')
elif PANDAS_AVAILABLE:
    try:
        report_df = pd.DataFrame(results)
        # display() asegura renderizado en Jupyter
        try:
            from IPython.display import display
            display(report_df)
        except Exception:
            print(report_df.to_string(index=False))
    except Exception as ex:
        print(f'Error al generar DataFrame: {ex}')
        for item in results:
            print(item)
else:
    for item in results:
        print(item)

                           ORIGEN                           
  Host : 172.19.0.9:1521
  User : USRSEGURIDAD
  SID  : INABIF02

                          DESTINO                           
  Host : 172.19.0.9:1521
  User : USRSEGURIDAD
  SID  : INABIF02

Acciones solicitadas: ['6']
Total resultados: 1



,action,status,tables,columns,output_file,message
0,EXPORT_TABLES_MD,OK,1112,22512,C:\work-space\sigeif-app\sigeif_ssi_inabif_pls...,Export completo generado.
